# AI-Based Crop Health Monitoring System
## Exploratory Data Analysis, Model Benchmarking & Explainable AI (Grad-CAM)
**Author:** Senior AI/ML & Deep Learning Engineer  
**Objective:** End-to-end investigation of plant foliar pathology, microclimate biophysics, convolutional neural networks, and gradient-weighted saliency maps.

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

# Ensure project root in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import CropHealthCNN, get_mobilenet_v2_model
from src.environmental_analysis import generate_agronomic_dataset, calculate_vpd
from src.predict import get_predictor

print("Environment initialized successfully. PyTorch version:", torch.__version__)

### 1. Dataset Verification & Class Breakdown
We inspect the verified PlantVillage dataset splits across the 6 target classes.

In [ ]:
with open(PROJECT_ROOT / "outputs" / "reports" / "evaluation_report.json", "r") as f:
    eval_data = json.load(f)

dataset_stats = eval_data["dataset_statistics"]
print(json.dumps(dataset_stats, indent=2))

classes = dataset_stats["classes"]
counts = [80] * len(classes) # 56 train, 12 val, 12 test

plt.figure(figsize=(10, 4))
sns.barplot(x=[c.replace('___', ' / ').replace('_', ' ') for c in classes], y=counts, palette='viridis')
plt.title('Verified PlantVillage Class Distribution (Balanced 80 samples/class)')
plt.ylabel('Image Count')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 2. Microclimate Biophysical Analysis: Vapor Pressure Deficit (VPD)
We analyze how Temperature and Humidity interact to generate VPD (kPa), determining pathogen risk.

In [ ]:
df_env = generate_agronomic_dataset(n_samples=1000, seed=42)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(data=df_env, x='temperature', y='humidity', hue='risk_category', palette='Set1', alpha=0.7)
plt.title('Microclimate Pathogen Risk Envelopes')
plt.xlabel('Temperature (°C)')
plt.ylabel('Relative Humidity (%)')

plt.subplot(1, 2, 2)
sns.histplot(df_env['vpd'], kde=True, color='teal')
plt.title('Vapor Pressure Deficit (VPD in kPa) Distribution')
plt.xlabel('VPD (kPa)')
plt.tight_layout()
plt.show()

### 3. Model Parameterization and Benchmark Comparison
Compare parameter counts and test performance between Custom CropHealthCNN and MobileNetV2.

In [ ]:
cnn = CropHealthCNN(num_classes=6)
mobilenet = get_mobilenet_v2_model(num_classes=6, pretrained=False)

cnn_params = sum(p.numel() for p in cnn.parameters())
mb_params = sum(p.numel() for p in mobilenet.parameters())

print(f"Custom CropHealthCNN Parameter Count: {cnn_params:,}")
print(f"MobileNetV2 Parameter Count:         {mb_params:,}")

comparison = eval_data["model_comparison"]
print(f"\nTest Accuracy Winner: {comparison['winner']}")
print(f"Custom CNN Test Acc:  {comparison['custom_cnn']['accuracy']*100:.2f}%")
print(f"MobileNetV2 Test Acc: {comparison['mobilenet_v2']['accuracy']*100:.2f}%")

### 4. Explainable AI: Grad-CAM Saliency Analysis
Run Grad-CAM on a test specimen and visualize lesion attention.

In [ ]:
predictor = get_predictor()
sample_path = PROJECT_ROOT / "dataset" / "test" / "Apple___Black_rot" / "Apple___Black_rot_004.jpg"

if sample_path.exists():
    res = predictor.predict(str(sample_path), model_type="mobilenet", generate_cam=True)
    print(f"Predicted Class: {res['display_name']} ({res['confidence']*100:.2f}%)")
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(res['gradcam']['original_image'])
    axes[0].set_title('1. Original Leaf Specimen')
    axes[0].axis('off')
    
    axes[1].imshow(res['gradcam']['heatmap_image'])
    axes[1].set_title('2. Grad-CAM Activation Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(res['gradcam']['overlay_image'])
    axes[2].set_title('3. Pathology Saliency Overlay')
    axes[2].axis('off')
    
    plt.suptitle(f"Foliar Attention: {res['display_name']}", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()